In [ ]:
#Loads scripts, libraries, etc.
print('Hello, lets get started')
%matplotlib widget
%load_ext autoreload
%autoreload 2
import sys
import os

# Ensure SGM_Beamline_Analysis is always on sys.path regardless of launch directory
candidate_dirs = [
    os.path.abspath('SGM_Beamline_Analysis'),
    os.path.abspath('.'),
    os.path.abspath(os.path.join(os.getcwd(), 'SGM_Beamline_Analysis')),
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
]
for c_dir in candidate_dirs:
    if os.path.exists(c_dir) and c_dir not in sys.path:
        sys.path.insert(0, c_dir)

import importlib
import PyMca5
from analyze_sgm_bsky_data import analyze_sgm_bsky_data, browse_beamline_data
from analyze_sgm_bsky_data_batch import analyze_sgm_bsky_data_batch
from alignment_utils import interactive_roll_align, visualize_stitching_overlap
from sum_mcc1_spectra import sum_mcc1_spectra
from save_pymca_stack_h5 import save_pymca_stack_h5
from calibrate_sdd_xrf import run_calibration, SDDCalibrationGUI
import sdd_calibration_utils as calib_utils
from stitching_utils import stitch_quadrant_maps, interactive_stitching_trim
from skimage import restoration

from IPython.display import display
from plot_sgm_bsky_data import plot_sgm_bsky_data
from plot_sgm_bsky_data_batch import plot_sgm_bsky_data_batch
from pca_xanes_analysis import pca_xanes_analysis, run_pca_all_detectors
from interactive_cluster_merger import interactive_cluster_merger
from cluster_xanes_analysis import run_clustering_all_detectors, cluster_xanes_analysis
from save_pymca_4d_stack_h5 import save_pymca_4d_stack_h5
from pca_stack_analysis import pca_stack_analysis
from cluster_pca_spectra import cluster_pca_spectra
from plot_2d_rixs_map import plot_incident_vs_emission_2d
from plot_selected_emission_spectra import plot_selected_emission_spectra

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
os.environ["OMP_NUM_THREADS"] = "2"

print('Done loading libraries')

In [ ]:
# For calibrating XRF spectra for each SDD
# Saves a Json file that can be applied to all XRD spectra in plotting
# To use, check box "Enable SDD Energy Calib" in the plot_sgm_bsky_data and then refresh plot
# Should only need to be done when conditions change on the beamline but could be done for each experiment
run_calibration()

In [ ]:
# Loads multiple stacks or maps in a batch for analysis
# Choose to either select a directory or multi-select individual files
# Summary of the stack shown in console
sgm_data_batch = analyze_sgm_bsky_data_batch()


In [ ]:
# Maps the fluorescence and sample TEY signal.  Can select which image for the map.
# Plots the fluorescence and TEY spectra from a stack- saves Raw and Normalized XANES spectra for each SDD, I0 and sample TEY as a csv file.
# Saves XRF spectra for each SDD as a csv file. Saves images (i.e., maps) for each SDD.
# Can change t39M14239
# The ROI from the XRF spectrum to select the edge of interest. Each bin  about 10 eV.
# Can change the area selected on the image by dragging a rectangle or making a polygon over the area of interest. Can toggle between the two. 
# Note that the area selected will be where the spectra are derived from and saves in the csv file.
# To see the changes in the spectra need to click on the "Refresh Plots" button.
# Can use the internal I0 from the Au mesh (mcc1) or load an external I0 from a csv file for normalization- allows for which column to select
# MCC1 is the I0 from Au mesh before the KB mirrors, MCC2 is from a photodiode behind the sample (only recorded when no sample in the pathway), 
# which could be used as and Io, , MCC3 auxillary, MCC4 is TEY signal from the sample. 
# Can either show lines with or withput individual data points
# Can zoom in on spectra using tools on left.   
# Mouse over an image or plot and get the X and Y coordinates.
# Automatically adjusts the contrast for each image.  
# Plot as a scatter map (usual way) or as a heatmap.
# The default ROI on the XRF spectra is 0-255.  Drag left and right edges on the XRF spectra to select the ROI for the element of interest.
# Each bin is about 10 eV. 
# Calibrate the energy by adding an energy shift value - energy_shift = 0 as default.
# To add detailed information to the header change use_full_metadata = True.
# Can display spectra with the data points or not- show_markers = False spectra plotted without data points shown
# Can change the display of the images, selecting grey scale or color

# Auto-fit the first spectrum stack and apply its energy ROI to the whole batch.
# Select Yes when asked to apply SDD calibration. xrf_roi is the fallback if fitting fails.
plot_sgm_bsky_data_batch(sgm_data_batch,  auto_element_roi=True)
#plot_sgm_bsky_data(sgm_data, as_scatter_plot=True, use_color=False, fixed_roi=False, xrf_roi=[1680,1830], channel_roi =[150,180])

In [ ]:
# Determines the principal components for each SDD and the average of the SDDs
# Must first save the file using the 'Save XANES Spectra for PCA/CA" button in the plotting
# Displays the eigenimages and vectors for all SDDs
# Select the number of components
pca_results=run_pca_all_detectors(saved_pymca_stack, n_components=10)

In [ ]:
# For all detectors
cluster_results=run_clustering_all_detectors(pca_results, use_full_metadata=False, n_clusters=7)

In [ ]:
#For all detectors
interactive_cluster_merger(cluster_results)

In [ ]:
# Same other useful scripts

In [ ]:
# Adds the I0 from the Au mesh (mcc1) from a number of scans
# The reason is to increase the S/N ratio of the I0, thus increasing the S/N of the normalized spectrum
sum_mcc1_spectra()

In [ ]:
# For calibrating XRF spectra for each SDD
# Saves a Json file that can be applied to all XRD spectra in plotting
# To use, check box "Enable SDD Energy Calib" in the plot_sgm_bsky_data and then refresh plot
# Should only need to be done when conditions change on the beamline but could be done for each experiment
run_calibration()

In [ ]:
# For single detector
# Determined the principal components for each SDD, or for the sum or average of eathe detectors
# Select the SDD and number of components.
# Displays the eigenimages and vectors

pca_results=pca_xanes_analysis(saved_pymca_stack, dataset_name ='sdd3', n_components =10)

In [ ]:
# For single detector
# SDD selected should be the same as thefor the pca_xanes_analysis
# Plots cluster maps and spectra for each detector and number of clusters selected
cluster_results=cluster_xanes_analysis(pca_results, dataset_name='sdd3', use_full_metadata=False,  n_clusters=7)

In [ ]:
# For a single detector
# Allows for pixels to be exclude from the spectra
# SDD selected should be the same as thefor the pca_xanes_analysis
interactive_cluster_merger(cluster_results, dataset_name ='sdd3')